# Dataset-Preprocessing 

--- 
**Note**: 
This project uses data from https://www.kaggle.com/datasets/shriyashjagtap/fraudulent-e-commerce-transactions

---

Columns in Original Dataset:
    Transaction ID,
    Customer ID,
    Transaction Amount,
    Transaction Date,
    Payment Method,
    Product Category,
    Quantity,
    Customer Age,
    Customer Location,
    Device Used,
    IP Address,
    Shipping Address,
    Billing Address,
    Is Fraudulent,
    Account Age Days,
    Transaction Hour

---





#### Columns in Original Dataset:
    Transaction ID,
    Customer ID,
    Transaction Amount,
    Transaction Date,
    Payment Method,
    Product Category,
    Quantity,
    Customer Age,
    Customer Location,
    Device Used,
    IP Address,
    Shipping Address,
    Billing Address,
    Is Fraudulent,
    Account Age Days,
    Transaction Hour

###### (https://www.kaggle.com/datasets/shriyashjagtap/fraudulent-e-commerce-transactions)

## **Objective:**
### Clean raw data from kaggle and save processed arrays for all model notebooks to share 





### 1. Deciding what data to use

In [25]:
"""
1. Drop TransactionID 
2. Check if CustomerID has repeated values. 
    If not, we can drop it as it does not add value to the model.
"""
import pandas as pd

df = pd.read_csv('../data/Fraudulent_E-Commerce_Transaction_Data.csv')

# Drop TransactionID as it is not useful for modeling
df.drop(columns=['Transaction ID'], inplace=True)

# Check for duplicates in CustomerID
if df['Customer ID'].duplicated().any():
    print("CustomerID has repeated values. It may be useful for modeling.")
else:
    df.drop(columns=['Customer ID'], inplace=True)
    print("CustomerID has no repeated values. It can be dropped as it does not add value to the model.")

print(df.shape)
df.head()
list(df.columns)


CustomerID has no repeated values. It can be dropped as it does not add value to the model.
(1472952, 14)


['Transaction Amount',
 'Transaction Date',
 'Payment Method',
 'Product Category',
 'Quantity',
 'Customer Age',
 'Customer Location',
 'Device Used',
 'IP Address',
 'Shipping Address',
 'Billing Address',
 'Is Fraudulent',
 'Account Age Days',
 'Transaction Hour']

In [ ]:
import numpy as np

"""
3. Check if address == billing address. 
    If same, create 1, else 0. 
    Drop the original address columns.
4. To represent Payment Method, Category and Device
    we can use one-hot encoding.
5. Scale the numerical features (Transaction Amount, Time of Transaction) using StandardScaler.
6. Use StanddardScaler for Amount, Age, Quantity.

"""
# Check if address == billing address and create a new binary feature
df['Address Match'] = (df['Shipping Address'] == df['Billing Address']).astype(int)

# Convert time to cylindrical Features
df['Hour_Sin'] = np.sin(2 * np.pi * df['Transaction Hour'] / 24)
df['Hour_Cos'] = np.cos(2 * np.pi * df['Transaction Hour'] / 24)


Cols_to_drop = ['Shipping Address', 'Billing Address', 
                'Transaction Hour', 'Transaction Date', 
                'IP Address']

df.drop(columns=Cols_to_drop, inplace=True)


"""
One-hot Encoding for categorical features
drop_first = True to avoid multicollinearity

"""
categorical_features = ['Payment Method', 'Product Category', 'Device Used']
df = pd.get_dummies(df, columns=categorical_features, drop_first=True) 

if 'Customer Location' in df.columns:
    df.drop(columns=['Customer Location'], inplace=True)

print(f"Final shape: {df.shape}")
print(df.dtypes)

# print(df.columns.tolist)
# print(df.columns)
print(f"Final shape: {df.shape}")
print(df.dtypes)


Final shape: (1472952, 17)
Transaction Amount                  float64
Quantity                              int64
Customer Age                          int64
Is Fraudulent                         int64
Account Age Days                      int64
Address Match                         int64
Hour_Sin                            float64
Hour_Cos                            float64
Payment Method_bank transfer           bool
Payment Method_credit card             bool
Payment Method_debit card              bool
Product Category_electronics           bool
Product Category_health & beauty       bool
Product Category_home & garden         bool
Product Category_toys & games          bool
Device Used_mobile                     bool
Device Used_tablet                     bool
dtype: object
Final shape: (1472952, 17)
Transaction Amount                  float64
Quantity                              int64
Customer Age                          int64
Is Fraudulent                         int64
Account 

### 2. Data Splitting

- Train (70%) — normal transactions only
- Validation (15%) — mix of normal + fraud
- Test (15%) — mix of normal + fraud

In [27]:
from sklearn.model_selection import train_test_split

normal_data = df[df['Is Fraudulent'] == 0].drop(columns=['Is Fraudulent'])
fraud_data = df[df['Is Fraudulent'] == 1].drop(columns=['Is Fraudulent'])


""" 

Splitting the data into training, validation, and testing sets
Then combine the normal and fraudulent data for val and test

random_state is set to shuffle the data bfr splitting, 
    = 42 is set for reproducibility.


"""
# 30% set for testing, 70% training
normal_train, norm_temp = train_test_split(normal_data, test_size=0.3, random_state=42) 
# split the 30% into 15% validation and 15% testing
normal_val, normal_test = train_test_split(norm_temp, test_size=0.5, random_state=42) 

fraud_val, fraud_test = train_test_split(fraud_data, test_size=0.5, random_state=42)

# Combine the normal and fraudulent data for val and test
val_df = pd.concat([normal_val.assign(label=0), fraud_val.assign(label=1)]).sample(frac=1, random_state=42)
test_df = pd.concat([normal_test.assign(label=0), fraud_test.assign(label=1)]).sample(frac=1, random_state=42)

x_train = normal_train.values.astype('float32') # Exclude labels column (everything is same value) 
x_val = val_df.drop(columns=['label']).values.astype('float32') # drop label as label is the target variable
y_val = val_df['label'].values # target variable for val
x_test = test_df.drop(columns=['label']).values.astype('float32')
y_test = test_df['label'].values

print(f"Train: {x_train.shape}, Val: {x_val.shape}, Test: {x_test.shape}")
print(f"Val fraud ratio:  {y_val.mean():.2%}")
print(f"Test fraud ratio: {y_test.mean():.2%}")

Train: (979379, 16), Val: (246786, 16), Test: (246787, 16)
Val fraud ratio:  14.96%
Test fraud ratio: 14.96%


In [29]:
# Standardize the features using StandardScaler

from sklearn.preprocessing import StandardScaler
import joblib
import os

scaler = StandardScaler()

x_train = scaler.fit_transform(x_train)
x_val = scaler.transform(x_val)
x_test = scaler.transform(x_test)

os.makedirs('../models', exist_ok=True)
joblib.dump(scaler, '../models/scaler.pkl')


['../models/scaler.pkl']